# 09 — Convolution（1D 卷积）

## 背景

卷积是 CNN 的基础算子，也广泛用于音频处理、时序建模（WaveNet、Mamba 的 SSM 卷积等）。其核心特点是**权重共享**：同一组卷积核在不同位置滑动复用。

GPU 上实现卷积的难点：
1. **边界处理**：same padding 要求对越界位置补零，需要条件判断（用无分支写法避免 warp divergence）
2. **访存模式**：相邻输出位置的输入重叠（sliding window），朴素实现会大量重复加载同一数据

## 问题定义

**单通道 1D 卷积（same padding）**：
$$O[n, l] = \sum_{k=0}^{KL-1} X[n,\ l + k - \text{pad}] \times K[k], \quad \text{pad} = \lfloor(KL-1)/2\rfloor$$

**多输出通道 1D 卷积**：
$$O[n, l, f] = \sum_{k=0}^{KL-1} X[n,\ l+k-\text{pad}] \times K[k, f]$$

In [ ]:
import tilelang
import tilelang.language as T
import triton
import triton.language as tl
import torch

tilelang.disable_cache()
print("TileLang:", tilelang.__version__)
print("Triton:  ", triton.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A")

## PyTorch 参考实现

In [ ]:
# Single-channel
N_b, L, KL = 64, 1024, 7
X_1d  = torch.randn(N_b, L, dtype=torch.float16, device="cuda")
Kern  = torch.randn(KL,     dtype=torch.float16, device="cuda")

# Multi-channel
N_mc, L_mc, KL_mc, F = 16, 256, 7, 32
X_mc    = torch.randn(N_mc, L_mc, dtype=torch.float16, device="cuda")
Kern_mc = torch.randn(KL_mc, F,   dtype=torch.float16, device="cuda")

_arch_conv = torch.cuda.get_device_properties(0).gcnArchName

def ref_conv1d(X, K):
    pad = (K.shape[0] - 1) // 2
    if _arch_conv.startswith("gfx1151"):
        # gfx1151 (iGPU)：MIOpen 对小问题规模启动开销大。
        # unfold+matmul：单个融合内核，比 MIOpen 快 34%（0.035ms vs 0.052ms）
        Xp = torch.nn.functional.pad(X, (pad, pad))
        Xu = Xp.unfold(1, K.shape[0], 1)          # [N, L, KL]
        return (Xu.float() @ K.float().unsqueeze(-1)).squeeze(-1).half()
    return torch.conv1d(X.unsqueeze(1), K.view(1, 1, -1), padding=pad).squeeze(1)

def ref_conv1d_mc(X, K):
    # unfold+matmul：避免不同 GPU 上 MIOpen 算法差异。
    # Triton 和 TileLang 通过直接累加使用 float32 精度；
    # torch.conv1d 在 gfx1151 上可能使用 Winograd 等算法，
    # 导致与直接累加相比最大误差约 0.015（atol=1e-2 会失败）。
    # unfold+matmul 与下方两个 GPU kernel 的直接累加语义一致。
    pad = (K.shape[0] - 1) // 2
    Xp = torch.nn.functional.pad(X, (pad, pad))
    Xu = Xp.unfold(1, K.shape[0], 1)              # [N, L, KL]
    return (Xu.float() @ K.float()).half()          # [N, L, F]

O1d_ref = ref_conv1d(X_1d, Kern)
Omc_ref = ref_conv1d_mc(X_mc, Kern_mc)
print(f"Single-ch: X{X_1d.shape} * K{Kern.shape} → O{O1d_ref.shape}")
print(f"Multi-ch:  X{X_mc.shape} * K{Kern_mc.shape} → O{Omc_ref.shape}")

## Triton 实现

每个 Triton program 负责输出张量的一个 `[BLOCK_N, BLOCK_L]` 子块：
- 外层循环 KL 次（卷积核长度），每次偏移 `src = l + k - pad`
- 边界 mask：`(src >= 0) & (src < L)`，越界位置用 `other=0.0`
- 卷积核权重 `kv` 为标量，直接广播

In [ ]:
@triton.jit
def _conv1d_kernel(X_ptr, K_ptr, O_ptr, N, L, KL, pad,
                   BN: tl.constexpr, BL: tl.constexpr):
    pid_n = tl.program_id(0);  pid_l = tl.program_id(1)
    ns  = pid_n * BN + tl.arange(0, BN)
    ls  = pid_l * BL + tl.arange(0, BL)
    acc = tl.zeros([BN, BL], dtype=tl.float32)
    for k in range(KL):
        src  = ls + k - pad
        mask = (ns[:, None] < N) & (src[None, :] >= 0) & (src[None, :] < L)
        x    = tl.load(X_ptr + ns[:, None] * L + src[None, :], mask=mask, other=0.0)
        kv   = tl.load(K_ptr + k)
        acc  = acc + x.to(tl.float32) * kv.to(tl.float32)
    mask_o = (ns[:, None] < N) & (ls[None, :] < L)
    tl.store(O_ptr + ns[:, None] * L + ls[None, :], acc.to(tl.float16), mask=mask_o)

@triton.jit
def _conv1d_mc_kernel(X_ptr, K_ptr, O_ptr, N, L, KL, F, pad,
                      BN: tl.constexpr, BL: tl.constexpr):
    pid_n = tl.program_id(0);  pid_l = tl.program_id(1);  f = tl.program_id(2)
    ns  = pid_n * BN + tl.arange(0, BN)
    ls  = pid_l * BL + tl.arange(0, BL)
    acc = tl.zeros([BN, BL], dtype=tl.float32)
    for k in range(KL):
        src  = ls + k - pad
        mask = (ns[:, None] < N) & (src[None, :] >= 0) & (src[None, :] < L)
        x    = tl.load(X_ptr + ns[:, None] * L + src[None, :], mask=mask, other=0.0)
        kv   = tl.load(K_ptr + k * F + f, mask=f < F, other=0.0)
        acc  = acc + x.to(tl.float32) * kv.to(tl.float32)
    mask_o = (ns[:, None] < N) & (ls[None, :] < L) & (f < F)
    tl.store(O_ptr + (ns[:, None] * L + ls[None, :]) * F + f,
             acc.to(tl.float16), mask=mask_o)

BN_t, BL_t = 4, 64

def triton_conv1d(X, K):
    N_, L_ = X.shape;  KL_ = K.shape[0];  pad = (KL_ - 1) // 2
    O = torch.empty_like(X)
    _conv1d_kernel[(triton.cdiv(N_, BN_t), triton.cdiv(L_, BL_t))](
        X, K, O, N_, L_, KL_, pad, BN=BN_t, BL=BL_t)
    return O

def triton_conv1d_mc(X, K):
    N_, L_ = X.shape;  KL_, F_ = K.shape;  pad = (KL_ - 1) // 2
    O = torch.empty(N_, L_, F_, dtype=X.dtype, device=X.device)
    _conv1d_mc_kernel[(triton.cdiv(N_, BN_t), triton.cdiv(L_, BL_t), F_)](
        X, K, O, N_, L_, KL_, F_, pad, BN=BN_t, BL=BL_t)
    return O

ok1 = torch.allclose(O1d_ref, triton_conv1d(X_1d, Kern), atol=1e-2)
ok2 = torch.allclose(Omc_ref, triton_conv1d_mc(X_mc, Kern_mc), atol=1e-2)
print("Triton single-ch:", "✅" if ok1 else "❌", "  multi-ch:", "✅" if ok2 else "❌")

## TileLang 实现

TileLang 中 `T.if_then_else((src >= 0) & (src < L), X[n, src], T.float16(0))` 实现无分支 zero padding，与 Triton 的 `mask + other=0.0` 语义等价。

In [ ]:
# ── GPU-adaptive config ──────────────────────────────────────────────────────
# gfx1100 (RX 7900 XTX): BN=4, BL=64, threads=256 (single-ch); BN=4, BL=32, BF=32 (multi-ch)
# gfx1201 (R9700): BN=4, BL=64, threads=128 (single-ch) — threads=256 is too many for N=64;
#   fewer blocks (16 vs 32 with TH=256) means better per-block work, 0.004ms vs 0.014ms.
# gfx1151 (Radeon 8060S): RDNA3.5 iGPU — same tile, compute-bound inner KL loop dominates
arch = torch.cuda.get_device_properties(0).gcnArchName

# single-ch threads: gfx1201 uses 128 (optimal for N_b=64 small grid)
TH_1D = 128 if arch.startswith("gfx1201") else 256
BN_1D, BL_1D = 4, 64
BN_MC, BL_MC, BF_MC = 4, 32, 32

@tilelang.jit(pass_configs={
    tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
    tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
})
def tl_conv1d(X, K, BLOCK_N: int, BLOCK_L: int):
    N_, L_, KL_ = T.const("N, L, KL")
    fp16, fp32 = T.float16, T.float32
    X: T.Tensor((N_, L_), fp16);  K: T.Tensor((KL_,), fp16)
    O = T.empty((N_, L_), fp16)
    pad = (KL_ - 1) // 2
    with T.Kernel(N_ // BLOCK_N, L_ // BLOCK_L, threads=TH_1D) as (pid_n, pid_l):
        acc = T.alloc_fragment((BLOCK_N, BLOCK_L), fp32)
        T.clear(acc)
        for k in T.Serial(KL_):
            for i, j in T.Parallel(BLOCK_N, BLOCK_L):
                n   = pid_n * BLOCK_N + i
                l   = pid_l * BLOCK_L + j
                src = l + k - pad
                acc[i, j] = acc[i, j] + T.if_then_else(
                    (src >= 0) & (src < L_),
                    X[n, src].astype(fp32) * K[k].astype(fp32), fp32(0))
        for i, j in T.Parallel(BLOCK_N, BLOCK_L):
            O[pid_n * BLOCK_N + i, pid_l * BLOCK_L + j] = acc[i, j].astype(fp16)
    return O

@tilelang.jit(pass_configs={
    tilelang.PassConfigKey.TL_DISABLE_WARP_SPECIALIZED: True,
    tilelang.PassConfigKey.TL_DISABLE_TMA_LOWER: True,
})
def tl_conv1d_mc(X, K, BLOCK_N: int, BLOCK_L: int, BLOCK_F: int):
    N_, L_, KL_, F_ = T.const("N, L, KL, F")
    fp16, fp32 = T.float16, T.float32
    X: T.Tensor((N_, L_), fp16);  K: T.Tensor((KL_, F_), fp16)
    O = T.empty((N_, L_, F_), fp16)
    pad = (KL_ - 1) // 2
    with T.Kernel(N_ // BLOCK_N, L_ // BLOCK_L, F_ // BLOCK_F, threads=256) as (pid_n, pid_l, pid_f):
        acc = T.alloc_fragment((BLOCK_N, BLOCK_L, BLOCK_F), fp32)
        T.clear(acc)
        for k in T.Serial(KL_):
            for i, j, ff in T.Parallel(BLOCK_N, BLOCK_L, BLOCK_F):
                n   = pid_n * BLOCK_N + i
                l   = pid_l * BLOCK_L + j
                f   = pid_f * BLOCK_F + ff
                src = l + k - pad
                acc[i, j, ff] = acc[i, j, ff] + T.if_then_else(
                    (src >= 0) & (src < L_),
                    X[n, src].astype(fp32) * K[k, f].astype(fp32), fp32(0))
        for i, j, ff in T.Parallel(BLOCK_N, BLOCK_L, BLOCK_F):
            O[pid_n * BLOCK_N + i, pid_l * BLOCK_L + j, pid_f * BLOCK_F + ff] = acc[i, j, ff].astype(fp16)
    return O

k_1d = tl_conv1d.compile(N=N_b, L=L, KL=KL, BLOCK_N=BN_1D, BLOCK_L=BL_1D)
k_mc = tl_conv1d_mc.compile(N=N_mc, L=L_mc, KL=KL_mc, F=F, BLOCK_N=BN_MC, BLOCK_L=BL_MC, BLOCK_F=BF_MC)

ok1 = torch.allclose(O1d_ref, k_1d(X_1d.clone(), Kern.clone()), atol=1e-2)
ok2 = torch.allclose(Omc_ref, k_mc(X_mc.clone(), Kern_mc.clone()), atol=1e-2)
print("TileLang single-ch:", "✅" if ok1 else "❌", "  multi-ch:", "✅" if ok2 else "❌")

## 性能对比

In [ ]:
import matplotlib.pyplot as plt

WARMUP, REPEAT = 20, 200

def bench(fn, args, warmup=WARMUP, repeat=REPEAT):
    for _ in range(warmup): fn(*args)
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    for _ in range(repeat): fn(*args)
    e.record(); torch.cuda.synchronize()
    return s.elapsed_time(e) / repeat

ms_1d = [bench(ref_conv1d,       [X_1d, Kern]),
         bench(triton_conv1d,    [X_1d, Kern]),
         bench(k_1d,             [X_1d, Kern])]
ms_mc = [bench(ref_conv1d_mc,    [X_mc, Kern_mc]),
         bench(triton_conv1d_mc, [X_mc, Kern_mc]),
         bench(k_mc,             [X_mc, Kern_mc])]

labels = ["PyTorch", "Triton", "TileLang"]
colors = ["#4C72B0", "#55A868", "#C44E52"]

print("Single-ch:", {l: f"{ms:.4f} ms" for l, ms in zip(labels, ms_1d)})
print("Multi-ch: ", {l: f"{ms:.4f} ms" for l, ms in zip(labels, ms_mc)})

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, data, title in zip(axes, [ms_1d, ms_mc],
    [f"Single-channel Conv  (N={N_b}, L={L}, KL={KL}, {arch})",
     f"Multi-channel Conv   (N={N_mc}, L={L_mc}, KL={KL_mc}, F={F}, {arch})"]):
    bars = ax.bar(labels, data, color=colors, width=0.5, edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, data):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(data)*0.01,
                f"{val:.4f} ms", ha="center", va="bottom", fontsize=9)
    ax.set_ylabel("Latency (ms)"); ax.set_title(title)
    ax.set_ylim(0, max(data)*1.3); ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.show()